# 04 · Q2: Profitability and discounts
COO question: "We give discounts on almost everything but margins are thin. Which categories should we grow, fix or stop, and should we cut discounts?"

In [1]:
%run common.py
con = connect()

## 7.1 Revenue, profit and margin by year

In [2]:
con.sql("""
    SELECT order_year,
           SUM(revenue) AS revenue,
           SUM(item_profit) AS profit,
           ROUND(SUM(item_profit) / SUM(revenue) * 100, 1) AS margin_pct
    FROM order_items
    GROUP BY order_year
    ORDER BY order_year
""").df()

,order_year,revenue,profit,margin_pct
0,2015,"11,089,543.55","1,318,856.90",11.90
1,2016,"11,055,996.76","1,310,119.07",11.80
2,2017,"10,610,910.01","1,304,085.11",12.30
3,2018,"297,952.05","33,841.89",11.40


## 7.2 How much do loss-making items cost?

In [3]:
con.sql("""
    SELECT SUM(CASE WHEN item_profit > 0 THEN item_profit ELSE 0 END) AS profit_from_good_items,
           SUM(CASE WHEN item_profit < 0 THEN item_profit ELSE 0 END) AS losses,
           SUM(item_profit) AS net_profit
    FROM order_items
""").df()

,profit_from_good_items,losses,net_profit
0,"7,850,450.32","-3,883,547.35","3,966,902.97"


## 7.3 Categories: Grow, Fix or Stop
Rule (company margin is 12%):
- **Grow:** revenue of $1M or more and margin of 12% or more
- **Fix:** revenue of $1M or more and margin below 12%
- **Stop / review:** revenue below $1M and margin below 8%
- **Keep:** everything else

In [4]:
con.sql("""
    SELECT category_name,
           SUM(revenue) AS revenue,
           ROUND(SUM(item_profit) / SUM(revenue) * 100, 1) AS margin_pct,
           CASE WHEN SUM(revenue) >= 1000000 AND SUM(item_profit) / SUM(revenue) >= 0.12 THEN 'Grow'
                WHEN SUM(revenue) >= 1000000 THEN 'Fix'
                WHEN SUM(item_profit) / SUM(revenue) < 0.08 THEN 'Stop'
                ELSE 'Keep' END AS action
    FROM order_items
    GROUP BY category_name
    ORDER BY revenue DESC
""").df()

,category_name,revenue,margin_pct,action
0,Fishing,"6,226,935.29",12.10,Grow
1,Cleats,"3,982,856.50",12.40,Grow
2,Camping & Hiking,"3,700,783.63",11.60,Fix
3,Cardio Equipment,"3,320,250.70",11.50,Fix
4,Women's Apparel,"2,828,708.50",12.40,Grow
5,Water Sports,"2,798,044.18",11.60,Fix
6,Men's Footwear,"2,598,494.42",12.00,Grow
7,Indoor/Outdoor Games,"2,596,454.02",12.30,Grow
8,Shop By Sport,"1,177,185.63",11.00,Fix
9,Computers,"595,395.00",11.70,Keep


 ## 7.4 Do bigger discounts sell more?
Average quantity per item in each discount band.

In [5]:
con.sql("""
    SELECT discount_band, ROUND(AVG(quantity), 2) AS avg_quantity
    FROM order_items
    GROUP BY discount_band
    ORDER BY discount_band
""").df()

,discount_band,avg_quantity
0,0-5%,2.13
1,11-15%,2.13
2,16-25%,2.13
3,6-10%,2.13


## 7.5 Export data for Excel
Two small tables for the Excel pivot table and discount model.

In [6]:
con.sql("SELECT category_name, SUM(revenue) AS revenue, SUM(item_profit) AS profit FROM order_items GROUP BY category_name") \
   .df().to_csv('../excel/categories.csv', index=False)

con.sql("SELECT discount_rate, SUM(gross_sales) AS gross_sales FROM order_items WHERE order_year = 2017 GROUP BY discount_rate ORDER BY discount_rate") \
   .df().to_csv('../excel/discounts_2017.csv', index=False)

print('Saved')

Saved


## Answer for the COO

- Bigger discounts don't sell more. A 10% cap would have added about $367K to 2017 profit.
- Grow Fishing, Cleats and the other big high-margin categories; fix pricing in Camping & Hiking and similar; review small categories under 8% margin.
- 1 in 5 items is sold at a loss, costing $3.9M. Finance should check product pricing and costs.